In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import math

### Import building footprints and wards

In [2]:
gdf_boundaries_v2 = gpd.read_file(r"/home/julian/notebooks/socio-econometrics/Patan_Wards.geojson")
buildings_gdf = gpd.read_parquet(r"/home/julian/notebooks/socio-econometrics/Patan _building_footprint.parquet")

In [3]:
len(buildings_gdf)

182313

**Adapting cells to "Churu" Format**

The main script was developed based on a slighly different data structure, thus, the need to adapt it to that format.

In [4]:
gdf_boundaries_v2 = gdf_boundaries_v2[gdf_boundaries_v2['Level']== "WARD"]
gdf_boundaries_v2['Ward_Number_str'] = (
    gdf_boundaries_v2['Name']
    .str.extract(r'WARD NO\.-(\d+)', expand=False)
)
gdf_boundaries_v2['Name'] = gdf_boundaries_v2['Ward_Number_str'].astype(int)
gdf_boundaries_v2['Name'].astype(str)
gdf_boundaries_v2 ['Ward_Number_str'].astype(str)
gdf_boundaries = gdf_boundaries_v2.copy()

Filtering out non-residential data and reprojecting to a known CRS.

In [5]:
buildings = buildings_gdf[buildings_gdf["prediction"] == "Residential"].copy()
other_buildings= buildings_gdf[buildings_gdf["prediction"] != "Residential"].copy()
buildings["geometry2"] = buildings_gdf["geometry"]
buildings = gpd.GeoDataFrame(
    buildings,
    geometry=gpd.points_from_xy(buildings["longitude"], buildings["latitude"]),
    crs="EPSG:4326"
)

In [6]:
len(buildings)

177818

Performing geospatial operation to join ward information to Buildings

In [7]:
joined = gpd.sjoin(buildings, gdf_boundaries, how="left", predicate="within")
joined_wo_nan = joined[joined['Name'].notna()]
buildings = joined_wo_nan

Importing excel file with population counts. Same procedure to adapt data.

In [8]:
#file from MHT
populationv2 = pd.read_excel(r"/home/julian/notebooks/socio-econometrics/ward_patan2011_final.xlsx")
populationv2 = populationv2[populationv2['Level']== "WARD"]
populationv2['Ward_Number_str'] = (
    populationv2['Name']
    .str.extract(r'WARD NO\.-(\d+)', expand=False)
)
populationv2['Name'] = populationv2['Ward_Number_str'].astype(int)
populationv2['Name'].astype(str)
populationv2 ['Ward_Number_str'].astype(str)
populationv2 ['WardNo'] = populationv2['Name']
population = populationv2.copy()

In [9]:
#total ground floor area
total_gfa = buildings['gfa_in_meters'].sum()
total_gfa

np.float64(4324165.11017)

In [10]:
#average gfa per inhabitant 
total_population = population["Pop_2025"].sum()
average_area_per_inhabitant = total_gfa / total_population
average_area_per_inhabitant

np.float64(22.295786175588827)

In [11]:
buildings["inhabitants_whole_churu"] = (buildings["gfa_in_meters"] / total_gfa) * total_population
buildings['Name'].astype(int)
buildings['Name'].astype(str)

/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


0          7.0
3          4.0
6          1.0
13         8.0
14         4.0
          ... 
182142     1.0
182149    11.0
182165     1.0
182178     6.0
182179     1.0
Name: Name, Length: 36048, dtype: object

It distributes a total population among all the buildings in a district (a "ward"), with one critical rule: you can't have fractions of a person. It must assign whole numbers (integers) that add up perfectly to the ward's total population, and it needs a fair way to decide who gets the "leftover" people from the decimal remainders.

In [12]:
len(buildings)

36048

In [13]:
#without decimal numbers
buildings = buildings.copy()
buildings["inhabitants_with_integer_estimate"] = np.nan

for ward in range(1, 15):
    try:
        mask = buildings["Name"].astype(int) == ward
    except:
        mask = buildings["Name"].astype(str) == str(ward)
    
    df_ward = buildings.loc[mask].copy()
    
    try:
        pop_row = population[population["WardNo"].astype(int) == ward]
    except:
        pop_row = population[population["WardNo"].astype(str) == str(ward)]
    
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    if total_gfa == 0:
        continue

    df_ward["ROcc"] = (df_ward["gfa_in_meters"] / total_gfa) * total_pop
    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]

    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1.0)

    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1

    buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values


It does the same, but addst an informal_multiplier in buildings that are classified as "informal".

In [14]:
#without decimal numbers + informal settlements constant
buildings = buildings.copy()
buildings["inhabitants_with_integer_informal"] = np.nan

informal_multiplier = 4

for ward in range(1, 15):
    # mask = buildings["Name"].astype(str) == str(ward)
    # df_ward = buildings.loc[mask].copy()
    # pop_row = population[population["WardNo"] == ward]
    try:
        mask = buildings["Name"].astype(int) == ward
    except:
        mask = buildings["Name"].astype(str) == str(ward)
    
    df_ward = buildings.loc[mask].copy()
    #print(len(df_ward))
    
    try:
        pop_row = population[population["WardNo"].astype(int) == ward]
    except:
        pop_row = population[population["WardNo"].astype(str) == str(ward)]
    
           
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    #print(total_gfa)
    if total_gfa == 0:
        continue
    
    df_ward["weight"] = np.where(df_ward["settlement_clasification"].str.lower() == "informal", informal_multiplier, 1)
    df_ward["weighted_gfa"] = df_ward["gfa_in_meters"] * df_ward["weight"]
    total_weighted_gfa = df_ward["weighted_gfa"].sum()
    

    df_ward["ROcc"] = (df_ward["weighted_gfa"] / total_weighted_gfa) * total_pop
    

    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]
    
    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1)
    
    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1
    

    buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values


In [15]:
#adding back geometry + nonres buildings
buildings['geometry'] = buildings['geometry2'] 
merged = pd.concat([buildings, other_buildings], ignore_index=True)
merged_gdf = gpd.GeoDataFrame(merged, geometry='geometry')
merged_gdf = merged_gdf.to_crs(epsg=4326)
merged_gdf.to_parquet("Patan_population_breakdown_in_wards.parquet")

In [23]:
print(population['Pop_2025'].sum())
print(merged_gdf["inhabitants_with_integer_estimate"].sum())
print(merged_gdf["inhabitants_with_integer_informal"].sum())

193945.39739999996
193944.0
193944.0


In [16]:
buildings_out_of_wards = joined[~joined['Name'].notna()]

In [24]:
# Ensure CRS is consistent
for gdf in [buildings, other_buildings, buildings_out_of_wards]:
    gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)

# Merge all subsets 
merged = pd.concat([buildings, other_buildings, buildings_out_of_wards], ignore_index=True)
merged = merged.drop_duplicates(subset='geometry')

# Verify count and save
print("Total merged buildings:", len(merged))

output = "Patan_buildings_with_population.parquet"
merged.to_parquet(output, index=False, engine="pyarrow")

print(f"Saved all buildings to: {output}")

/usr/local/lib/python3.12/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Total merged buildings: 182313
Saved all buildings to: Patan_buildings_with_population.parquet
